# 📊 서울시 경찰서별 범죄 발생 및 검거 현황 데이터 분석
> **데이터 출처**: 서울 열린데이터광장 (Seoul Open Data)  
> **분석 파일**: `data/02. crime_in_Seoul.csv`

---

### 💡 학습 및 실습 목표
1. **한글 인코딩(`cp949`)과 천 단위 쉼표(`thousands=','`) 처리**:
   - 한글 윈도우 환경에서 생성된 CSV의 인코딩 문제(`cp949`/`euc-kr`) 해결
   - 수치에 포함된 콤마(예: `1,395`)를 제거하고 정수형(`int64`)으로 자동 변환
2. **데이터 구조 탐색 (EDA)**:
   - 서울시 31개 관할 경찰서별 5대 범죄(살인, 강도, 강간, 절도, 폭력) 발생 및 검거 건수 점검
   - 결측치 유무 및 데이터 타입 확인


In [23]:
# [셀 의도] 데이터 조작 및 분석을 위한 라이브러리를 불러옵니다.
import pandas as pd
import numpy as np

print(f"✅ 라이브러리 불러오기 완료! (Pandas 버전: {pd.__version__})")

✅ 라이브러리 불러오기 완료! (Pandas 버전: 3.0.6)


## 1. 범죄 CSV 데이터 파일 불러오기 (`pd.read_csv`)

### 📌 데이터 로드 시 필수 옵션
- **`encoding='cp949'`**: 파일 내 한글이 깨지지 않도록 인코딩 지정
- **`thousands=','`**: `1,395`, `2,056` 등의 쉼표 포함 숫자를 문자열이 아닌 정수형(`int64`)으로 로드


In [24]:
# [셀 의도] CSV 파일을 인코딩과 천 단위 쉼표 옵션을 적용하여 로드합니다.
csv_path = 'data/02. crime_in_Seoul.csv'

crime_seoul = pd.read_csv(csv_path, encoding='cp949', thousands=',')

print(f"✅ '{csv_path}' 불러오기 성공! (행: {crime_seoul.shape[0]}개, 열: {crime_seoul.shape[1]}개)")
crime_seoul.head()

✅ 'data/02. crime_in_Seoul.csv' 불러오기 성공! (행: 31개, 열: 11개)


,관서명,살인 발생,살인 검거,강도 발생,강도 검거,강간 발생,강간 검거,절도 발생,절도 검거,폭력 발생,폭력 검거
0,중부서,2,2,3,2,105,65,1395,477,1355,1170
1,종로서,3,3,6,5,115,98,1070,413,1278,1070
2,남대문서,1,0,6,4,65,46,1153,382,869,794
3,서대문서,2,2,5,4,154,124,1812,738,2056,1711
4,혜화서,3,2,5,4,96,63,1114,424,1015,861


## 2. 데이터 구조 및 기본 정보 확인 (Data Inspection)

- `shape`: 서울시 31개 경찰서(31행), 11개 컬럼(열)
- `info()`: 모든 범죄 수치가 `int64` 정수형으로 정상 로드되었는지, 결측치가 없는지 점검


In [25]:
# [셀 의도] 데이터의 크기, 컬럼 목록 및 데이터 타입을 점검합니다.
print("=" * 60)
print(f"1. 데이터 크기: {crime_seoul.shape[0]}개 경찰서(행), {crime_seoul.shape[1]}개 컬럼(열)")
print("=" * 60)

print("\n2. 데이터 요약 정보 (.info()):")
crime_seoul.info()

1. 데이터 크기: 31개 경찰서(행), 11개 컬럼(열)

2. 데이터 요약 정보 (.info()):
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   관서명     31 non-null     str  
 1   살인 발생   31 non-null     int64
 2   살인 검거   31 non-null     int64
 3   강도 발생   31 non-null     int64
 4   강도 검거   31 non-null     int64
 5   강간 발생   31 non-null     int64
 6   강간 검거   31 non-null     int64
 7   절도 발생   31 non-null     int64
 8   절도 검거   31 non-null     int64
 9   폭력 발생   31 non-null     int64
 10  폭력 검거   31 non-null     int64
dtypes: int64(10), str(1)
memory usage: 2.8 KB


In [26]:
# 관서명 끝의 '서'를 제외하고 '서울 '과 ' 경찰서'를 붙여 변경
crime_seoul['관서명'] = ['서울 ' + name[:-1] + ' 경찰서' for name in crime_seoul['관서명']]

# 변경 결과 확인 (상위 5개)
crime_seoul[['관서명']].head()


,관서명
0,서울 중부 경찰서
1,서울 종로 경찰서
2,서울 남대문 경찰서
3,서울 서대문 경찰서
4,서울 혜화 경찰서


In [27]:
crime_seoul.head()

,관서명,살인 발생,살인 검거,강도 발생,강도 검거,강간 발생,강간 검거,절도 발생,절도 검거,폭력 발생,폭력 검거
0,서울 중부 경찰서,2,2,3,2,105,65,1395,477,1355,1170
1,서울 종로 경찰서,3,3,6,5,115,98,1070,413,1278,1070
2,서울 남대문 경찰서,1,0,6,4,65,46,1153,382,869,794
3,서울 서대문 경찰서,2,2,5,4,154,124,1812,738,2056,1711
4,서울 혜화 경찰서,3,2,5,4,96,63,1114,424,1015,861


In [28]:
!pip install googlemaps folium

In [29]:
import googlemaps
import numpy as np

# 1. 구글맵 API 클라이언트 등록 (본인의 API 키 입력)
import os
gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 또는 발급받은 API 키
gmaps = googlemaps.Client(key=gmaps_key)

# 2. 31개 경찰서의 위도, 경도, 도로명 주소를 담을 리스트
station_lat = []
station_lng = []
station_address = []

# 앞서 수정한 '관서명'(예: 서울중부경찰서)으로 좌표 검색
for name in crime_seoul['관서명']:
    # 구글맵 지오코딩 검색
    tmp = gmaps.geocode(name, language='ko')
    
    if tmp:
        # 정상 검색된 경우 좌표 및 주소 추출
        tmp_loc = tmp[0].get('geometry')
        station_lat.append(tmp_loc['location']['lat'])
        station_lng.append(tmp_loc['location']['lng'])
        station_address.append(tmp[0].get('formatted_address'))
    else:
        # 검색 실패 시 결측치 처리
        station_lat.append(np.nan)
        station_lng.append(np.nan)
        station_address.append(np.nan)

# 3. 데이터프레임에 컬럼으로 추가
crime_seoul['lat'] = station_lat
crime_seoul['lng'] = station_lng
crime_seoul['주소'] = station_address

# 상위 5개 결과 확인
crime_seoul[['관서명', 'lat', 'lng', '주소']].head()


,관서명,lat,lng,주소
0,서울 중부 경찰서,37.559904,126.979491,대한민국 서울특별시 중구 퇴계로 67
1,서울 종로 경찰서,37.571824,126.984153,대한민국 서울특별시 종로구 인사동5길 41
2,서울 남대문 경찰서,37.554758,126.973498,대한민국 서울특별시 중구 한강대로 410
3,서울 서대문 경찰서,37.560607,126.962611,대한민국 서울특별시 서대문구 충정로 13
4,서울 혜화 경찰서,37.571968,126.998957,대한민국 서울특별시 종로구 창경궁로 112-16


In [30]:
import folium

# 1. 서울 중심 좌표를 기준으로 지도 생성
seoul_map = folium.Map(
    location=[37.5502, 126.982],  # 서울 중심부 위도/경도
    zoom_start=11,                  # 기본 줌 레벨
    tiles='OpenStreetMap'          # 지도 타일 스타일
)

# 2. 31개 경찰서 위치에 마커 추가
for idx, row in crime_seoul.iterrows():
    # 위도, 경도 값이 존재하는 경우만 마커 생성
    if not np.isnan(row['lat']) and not np.isnan(row['lng']):
        folium.Marker(
            location=[row['lat'], row['lng']],
            popup=folium.Popup(f"<b>{row['관서명']}</b><br>{row.get('주소', '')}", max_width=250),
            tooltip=row['관서명'],  # 마우스 올렸을 때 표시
            icon=folium.Icon(color='blue', icon='info-sign')
        ).add_to(seoul_map)

# 3. 주피터 노트북에 인터랙티브 지도 출력
seoul_map


In [31]:
import folium
import json
import numpy as np

# 1. 서울시 25개 자치구 경계선(GeoJSON) 데이터 불러오기
geo_path = 'data/skorea_municipalities_geo_simple.json'
with open(geo_path, 'r', encoding='utf-8') as f:
    geo_data = json.load(f)

# 서울시 행정구역 코드('11'로 시작하는 25개 자치구)만 추출
seoul_geo = {
    'type': geo_data['type'],
    'features': [f for f in geo_data['features'] if f['properties']['code'].startswith('11')]
}

# 2. VS Code 뷰어 높이 0 방지를 위한 Figure 생성 (가로 100%, 세로 650px)
fig = folium.Figure(width='100%', height=650)

# 3. Google Maps(구글 로드맵) 타일을 배경으로 지도 생성
seoul_map = folium.Map(
    location=[37.5502, 126.982],  # 서울 중심 좌표
    zoom_start=11,
    tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',  # 구글맵 타일 URL
    attr='Google Maps'
)
fig.add_child(seoul_map)

# 4. 지도 위에 구 경계선(Polygon) 그리기
folium.GeoJson(
    seoul_geo,
    name='서울시 자치구 경계',
    style_function=lambda feature: {
        'fillColor': '#1a73e8',      # 구 내부 채우기 색상 (연한 블루)
        'color': '#0d47a1',          # 구 경계선 테두리 색상 (선명한 네이비)
        'weight': 2.5,               # 경계선 두께
        'fillOpacity': 0.08,         # 지도가 잘 투과되도록 투명도 설정
    },
    # 마우스 올렸을 때 구 이름(예: 강남구, 종로구) 툴팁 표시
    tooltip=folium.GeoJsonTooltip(
        fields=['name'],
        aliases=['자치구: '],
        style='font-size: 13px; font-weight: bold; background-color: white; padding: 5px; border-radius: 4px;'
    )
).add_to(seoul_map)

# 5. 31개 관할 경찰서 위치에 마커 표시 (경계선 위에서 잘 보이도록 붉은색 마커 사용)
for idx, row in crime_seoul.iterrows():
    if not np.isnan(row['lat']) and not np.isnan(row['lng']):
        folium.Marker(
            location=[row['lat'], row['lng']],
            popup=folium.Popup(f"<div style='width:180px'><b>{row['관서명']}</b><br><small>{row.get('주소', '')}</small></div>", max_width=250),
            tooltip=row['관서명'],
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(seoul_map)

# 6. 노트북 셀 안에 지도 바로 렌더링
fig
